# Cellpose-SAM 細胞ROI抽出 (Google Colab版)

研究室のGPU付きPCが故障したため、[Cellpose公式のColabノートブック](https://github.com/MouseLand/cellpose/blob/main/notebooks/run_Cellpose-SAM.ipynb)（Cellpose-SAM）をベースに、この研究室の環境に合わせて構成したノートブックです。

- Cellpose本体は最新の **Cellpose-SAM**（`models.CellposeModel`）を使用（細胞の直径指定は不要）
- 撮影データはZスタック（`_C001Z001` 〜 `_C001Z007` のようなZスライスtifが複数枚、撮影ごとにフォルダが分かれている）想定 → **Zスライスごとに個別にセグメンテーション・ROI作成**（Z投影はしない。ピントの良いZは後でFiji上で選ぶ）
- セグメンテーション結果は、元のtifと**同じフォルダ・同じファイル名の `.zip`**（ImageJ RoiSet形式）として保存 → 従来の `intensity_analysis.ijm` マクロがそのまま読み込める

## Cellposeの設定値（研究室PCでの設定に合わせています）

- `flow_threshold` = 0.1
- `cellprob_threshold` = 2.0
- 最小面積 100 px → Cellposeの `min_size` パラメータでそのまま指定
- 最大面積 500 px → Cellpose本体にはこの機能がないため、セグメンテーション後にこのノートブック側で後処理フィルタとして実装
- Max Brightness Ratio 2.5 → **Cellpose本体には該当する設定がなく、正確な計算式が未確認のため、デフォルトでは無効化したプレースホルダーとして実装しています**（下記パラメータ設定セル参照）。有効化する前に、元のツールでの正確な定義をご確認ください。

## 全体の流れ

1. (このノートブック) Google Drive上の親フォルダ以下を再帰的に探索し、すべての `.tif` / `.tiff`（＝各撮影フォルダの各Zスライス）を読み込む
2. (このノートブック) Zスライス1枚ずつをCellpose-SAMで自動セグメンテーションし、面積フィルタ等を適用する
3. (このノートブック) セグメンテーション結果を、元のtifと同じフォルダ・同じファイル名の `.zip`（ImageJ RoiSet形式）として保存する
4. (Fiji/ImageJ) 使うZを選び、その `画像名.zip` を対応する画像と一緒に開き、ROI Managerでおかしい細胞のROIを削除し、背景ROIを最後に追加してzipを保存し直す
5. (Fiji/ImageJ) 従来の `intensity_analysis.ijm` マクロを実行して膜/細胞質の輝度解析を行う

## 事前準備

- 上部メニューの **「ランタイム」→「ランタイムのタイプを変更」→ハードウェアアクセラレータで GPU (T4など) を選択**してください。
- 撮影ごとのフォルダ（各フォルダの中にZスライスのtifが入っている）を、Google Drive上の1つの親フォルダにまとめておいてください。サブフォルダ構成のままで構いません（このノートブックが再帰的に探索します）。


In [ ]:
# GPUが割り当てられているか確認
!nvidia-smi


In [ ]:
# Cellposeのインストール（Cellpose-SAMを含む最新版。natsort/tifffile/roifile等も依存関係として入る）
!pip install -q cellpose


In [ ]:
# Google Driveをマウント
from google.colab import drive
drive.mount('/content/drive')


## パス設定・画像の探索

`INPUT_DIR` を、撮影フォルダ（Zスライスtifが入ったフォルダ）をまとめてある親フォルダに変更してください。例えば

```
INPUT_DIR/
  260703-WT-PMA-60min-1408-488/
    260703-WT-PMA-60min-1408-488_C001Z001.tif
    ...
    260703-WT-PMA-60min-1408-488_C001Z007.tif
  260703-他の撮影/
    ...
```

のような構成を想定し、`INPUT_DIR` 以下を再帰的に探索してすべての `.tif`/`.tiff` を1枚ずつ処理します（Z投影はせず、Zスライスごとに個別にROIを作ります）。

`OUTPUT_DIR` は ROI (`.zip`) の保存先です。Fijiのマクロは「画像と同じフォルダにある同名の `.zip`」を探すので、迷ったら `INPUT_DIR` と同じにしておくのが安全です（デフォルトでそうなっています。各tifと同じサブフォルダに保存されます）。

もし1つの撮影フォルダに複数チャンネル（例: `C001`, `C002`）のtifが混在していて、セグメンテーションに使いたいチャンネルが1つだけの場合は、`FILENAME_FILTER` にそのチャンネルを表す文字列（例: `"C001"`）を指定すると、そのチャンネルのファイルだけを処理できます。


In [ ]:
from pathlib import Path
from natsort import natsorted

# 撮影ごとのフォルダ（Zスライスtifが入っている）をまとめた親フォルダ
INPUT_DIR = Path("/content/drive/MyDrive/cellpose_input")
if not INPUT_DIR.exists():
    raise FileNotFoundError("INPUT_DIRが存在しません。パスを確認してください。")

# ROI (.zip) の保存先。Fijiのマクロが「各tifと同じフォルダの同名zip」を探すため、
# 基本はINPUT_DIRと同じにしておき、各tifと同じサブフォルダに保存する
OUTPUT_DIR = INPUT_DIR

# セグメンテーション結果を目視確認するための重ね合わせ画像 (QC画像) の保存先
# (INPUT_DIR以下のフォルダ構成をそのままミラーして保存する)
QC_DIR = OUTPUT_DIR / "qc_overlays"

# 複数チャンネルがあり、特定のチャンネルだけをセグメントしたい場合はファイル名に
# 含まれる文字列を指定する（例: "C001"）。すべて処理する場合は None のままでよい。
FILENAME_FILTER = None

QC_DIR.mkdir(parents=True, exist_ok=True)

# INPUT_DIR以下を再帰的に探索し、INPUT_DIRからの相対パスを集める
image_files = [
    p.relative_to(INPUT_DIR)
    for p in INPUT_DIR.rglob("*")
    if p.suffix.lower() in (".tif", ".tiff")
    and QC_DIR not in p.parents
    and (FILENAME_FILTER is None or FILENAME_FILTER in p.name)
]
image_files = natsorted(image_files, key=lambda p: str(p))

if len(image_files) == 0:
    raise FileNotFoundError("画像が見つかりませんでした。INPUT_DIRやFILENAME_FILTERを確認してください。")

print(f"{len(image_files)} 枚のZスライス画像が見つかりました（1枚ずつ個別にROIを作成します）")
for f in image_files:
    print(" -", f)


## モデルの準備

Cellpose-SAM（`models.CellposeModel`）を読み込みます。旧来のcyto2/cyto3のようなモデル選択や細胞直径の指定は不要です（初回実行時に重みがダウンロードされます）。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from cellpose import models, core, io, utils, plot

io.logger_setup()  # 進捗ログを表示する

if core.use_gpu() == False:
    raise ImportError("GPUにアクセスできません。「ランタイム」→「ランタイムのタイプを変更」でGPUを選択してください。")

model = models.CellposeModel(gpu=True)


## パラメータ設定

研究室のPCで使っていた設定値です。

- `FLOW_THRESHOLD` (0.1) / `CELLPROB_THRESHOLD` (2.0): Cellposeの `model.eval()` にそのまま渡します
- `MIN_SIZE` (最小面積 100 px): Cellposeの `min_size` パラメータにそのまま渡します（これ未満のROIは自動的に除外されます）
- `MAX_AREA` (最大面積 500 px): Cellpose本体にはこの機能がないため、セグメンテーション後に `remove_large_masks()` で後処理として除外します
- `TILE_NORM_BLOCKSIZE`: 画像内で明るさが大きく不均一な場合は100〜200程度に変更（デフォルト0は画像全体を一括で正規化）

**Max Brightness Ratio (2.5) について**: Cellpose本体には該当する設定がなく、元のツールでの正確な計算式が未確認です。誤ったROIの採否につながらないよう、デフォルトでは `APPLY_BRIGHTNESS_RATIO_FILTER = False` にして無効化しています。下のセルの `filter_by_brightness_ratio()` は「ROI内の最大輝度／平均輝度の比」という仮実装のプレースホルダーです。元のツールでの正確な定義を確認できたら、この関数を修正した上で `True` にしてください。


In [ ]:
FLOW_THRESHOLD = 0.1
CELLPROB_THRESHOLD = 2.0
MIN_SIZE = 100                # 最小面積(px)。Cellposeのmin_sizeにそのまま渡す
MAX_AREA = 500                # 最大面積(px)。後処理で除外する
TILE_NORM_BLOCKSIZE = 0       # 明るさが不均一な画像なら100-200程度に変更

# Max Brightness Ratio: 正確な定義が未確認のため、デフォルトでは無効
APPLY_BRIGHTNESS_RATIO_FILTER = False
MAX_BRIGHTNESS_RATIO = 2.5


## 面積フィルタ・輝度比フィルタ・ROI保存の関数


In [ ]:
import roifile


def remove_large_masks(masks, max_area):
    """max_area(px)を超えるROIを除去し、ラベルを振り直す"""
    labels, counts = np.unique(masks, return_counts=True)
    keep_labels = [l for l, c in zip(labels, counts) if l != 0 and c <= max_area]

    new_masks = np.zeros_like(masks)
    for new_id, old_id in enumerate(keep_labels, start=1):
        new_masks[masks == old_id] = new_id
    return new_masks


def filter_by_brightness_ratio(masks, img, max_ratio):
    """TODO: 元ツールでの正確な定義が未確認のプレースホルダー実装。
    現状の仮実装: ROI内の (最大輝度 / 平均輝度) が max_ratio を超えるROIを除外する。
    """
    labels = np.unique(masks)
    labels = labels[labels != 0]
    keep_labels = []
    for l in labels:
        pixels = img[masks == l]
        mean_val = pixels.mean()
        if mean_val <= 0:
            continue
        ratio = pixels.max() / mean_val
        if ratio <= max_ratio:
            keep_labels.append(l)

    new_masks = np.zeros_like(masks)
    for new_id, old_id in enumerate(keep_labels, start=1):
        new_masks[masks == old_id] = new_id
    return new_masks


def segment_and_filter(img):
    """Cellpose-SAMでセグメンテーションし、面積・輝度比フィルタを適用する"""
    masks, flows, styles = model.eval(
        img,
        batch_size=32,
        flow_threshold=FLOW_THRESHOLD,
        cellprob_threshold=CELLPROB_THRESHOLD,
        min_size=MIN_SIZE,
        normalize={"tile_norm_blocksize": TILE_NORM_BLOCKSIZE},
    )
    masks = remove_large_masks(masks, MAX_AREA)
    if APPLY_BRIGHTNESS_RATIO_FILTER:
        masks = filter_by_brightness_ratio(masks, img, MAX_BRIGHTNESS_RATIO)
    return masks, flows


def masks_to_imagej_roi_zip(masks, save_path):
    """Cellposeのマスクを画像1枚分のImageJ RoiSet (.zip) として保存する
    (cellpose.io.save_rois と同じロジックだが、ファイル名の末尾に "_rois" を付けず、
    Fijiマクロが期待する "画像名.zip" そのままの名前で保存する)
    """
    outlines = utils.outlines_list(masks)
    rois = []
    for i, outline in enumerate(outlines):
        if len(outline) < 3:
            continue
        roi = roifile.ImagejRoi.frompoints(outline, name=f"{i + 1:04d}")
        rois.append(roi)

    save_path = Path(save_path)
    if save_path.exists():
        save_path.unlink()
    roifile.roiwrite(str(save_path), rois, mode="w")
    return len(rois)


## パラメータのプレビュー（1枚だけ試す）

一括処理の前に、まず1枚（1つのZスライス）でセグメンテーション結果を確認します。細胞が大きすぎる/小さすぎる、うまく分割できていない等があれば、上のパラメータ設定セルを調整してこのセルを再実行してください。


In [ ]:
preview_file = image_files[0]
img = io.imread(str(INPUT_DIR / preview_file))
print(f"対象ファイル: {preview_file}")
print(f"画像shape: {img.shape}, dtype: {img.dtype}")

masks, flows = segment_and_filter(img)
print(f"フィルタ後: {masks.max()} 個の細胞を検出")

fig = plt.figure(figsize=(12, 5))
plot.show_segmentation(fig, img, masks, flows[0])
plt.tight_layout()
plt.show()


## 一括処理

`image_files` に含まれる全Zスライスに対して個別にセグメンテーション・フィルタ適用を行い、以下を保存します（Z投影はせず、1つのtifにつき1つのzipを作成します）。

- 元のtifと同じサブフォルダの `画像名.zip`: Fijiの `intensity_analysis.ijm` がそのまま読み込めるROI（細胞のみ、背景ROIは含みません）
- `QC_DIR` 以下（元と同じサブフォルダ構成）の `画像名_qc.png`: セグメンテーション結果を目視確認するための重ね合わせ画像

処理後、QC画像やFiji上でどのZが一番良いか確認し、使うZの `.zip` だけをFijiで開いて、おかしい細胞のROIを削除、背景ROIを最後に追加してから `.zip` を保存し直してください。


In [ ]:
results_summary = []

for rel_path in image_files:
    img_path = INPUT_DIR / rel_path
    img = io.imread(str(img_path))

    masks, flows = segment_and_filter(img)

    rel_dir = rel_path.parent
    basename = rel_path.stem

    # ROIは元のtifと同じサブフォルダに保存する（Fijiマクロの想定に合わせる）
    out_dir = OUTPUT_DIR / rel_dir
    out_dir.mkdir(parents=True, exist_ok=True)
    roi_path = out_dir / f"{basename}.zip"
    n_rois = masks_to_imagej_roi_zip(masks, roi_path)

    # 目視確認用の重ね合わせ画像を保存（フォルダ構成をミラーする）
    qc_out_dir = QC_DIR / rel_dir
    qc_out_dir.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(img, cmap="gray")
    ax.imshow(np.ma.masked_where(masks == 0, masks), cmap="jet", alpha=0.5)
    ax.set_title(f"{rel_path} ({n_rois} cells)")
    ax.axis("off")
    fig.savefig(qc_out_dir / f"{basename}_qc.png", dpi=100, bbox_inches="tight")
    plt.close(fig)

    results_summary.append((str(rel_path), n_rois))
    print(f"{rel_path}: {n_rois} 個のROIを保存 -> {roi_path}")

print("\n=== 完了 ===")
for rel_path, n in results_summary:
    print(f"{rel_path}: {n} cells")


## 次のステップ (Fiji/ImageJ側の作業)

1. Google Drive for desktopなどでPCと同期するか、Driveから直接ダウンロードして、各撮影フォルダに `画像名.tif` と `画像名.zip` が同じフォルダにあることを確認する
2. `QC_DIR`（`qc_overlays`フォルダ、元と同じサブフォルダ構成）内の `_qc.png` を撮影ごとに見比べて、一番ピントが合っている・セグメンテーションが綺麗なZを選ぶ（崩れている場合はパラメータを調整して再実行）
3. Fijiで選んだZの画像を開き、ROI Managerで対応する `画像名.zip` を読み込む
4. 明らかにおかしい細胞のROIを選択して削除する（`APPLY_BRIGHTNESS_RATIO_FILTER` を無効にしている間は、この手動チェックが輝度比フィルタの代わりになります）
5. 背景となる領域のROIを新規作成し、**ROI Managerの一番最後に追加**する（`intensity_analysis.ijm` は最後のROIを背景として扱う仕様）
6. ROI Managerの内容を `画像名.zip` として上書き保存する
7. 従来通り `intensity_analysis.ijm` マクロを実行する（使わない他のZのtif/zipは無視してよい）
